In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf 
import pyfixest as pf
from pyfixest.estimation import feols
import gc

In [2]:
# an alternative to panda's insert(args) since it throws an error if the column already exists
# same logic but instead of df.insert, we use insert_or_replace(df, args)
def insert_or_replace(df, position, name, values):
    if name in df.columns:
        df.drop(columns=name, inplace=True)
    df.insert(position, name, values)

# gives the position to insert columns after a specified column
from typing import cast

# for some reason pylance never lets me use .get_loc() + 1
def nextpos(df, col:str)->int:
    pos = cast(int, df.columns.get_loc(col))
    return pos + 1

# if i ever need to swap cols
def swapcolumns(df, col1:str, col2:str):
    columns = list(df.columns)
    new = columns.index(col1) 
    old = columns.index(col2)

    columns[new], columns[old] = columns[old], columns[new]

    df = df[columns]
    return df


In [3]:
# only reads necessary columns so that we dont store unnecessary stuff in ram
# market id is in the index itself
panel = pd.read_parquet("../data/pipeline/panel.parquet", columns=["p", "hours_to_end"])

### Mincer-Zarnowitz regression

Never ended up using this. Thought it was a pretty cool idea but it was pretty much entirely focused on calibration, and I couldn't see how I can tie this into the information story the paper tries to tell.

In [ ]:
def mincer_zarnowitz(df_og,
                    time_labels = [
                    "0-1 day",
                    "1-2 days",
                    "2-4 days",
                    "4-7 days",
                    "1-2 weeks",
                    "2-4 weeks",
                    "1-2 months",
                    "2-4 months",
                    ">4 months"]):
    
    # yes_outcomes was bool before which isnt accepted
    # so we'll make it 0/1
    df = df_og.copy()
    df["yes_outcomes"] = df["yes_outcomes"].astype(int)

    final_results = []
    final_tests = []

    # run a regression for every ttr bin given
    for time in time_labels:
        df_bin = df.loc[df.ttr_bin == time]

        # skip bins with no data in this cohort, rather than erroring
        # important for the cohorts
        if df_bin.empty:
            continue  

        # we only look at the last observation for each market
        # so each market contributes only 1 observation to the regression
        df_bin = df_bin.groupby("market_id").last()

        model = smf.ols('yes_outcomes ~ p', data = df_bin)
        market_ids = df_bin.index.get_level_values("market_id")

        results = model.fit(cov_type = "HC1")

        intercept = results.params["Intercept"]
        intercept_p = results.pvalues["Intercept"]

        slope = results.params["p"]
        slope_p = results.pvalues["p"]

        num_markets = market_ids.nunique()

        final = [intercept, intercept_p, slope, slope_p, num_markets]
        final_results.append(final)


        # testing null: intercept = 0 and slope = 1
        wald = results.wald_test("Intercept = 0, p = 1", use_f=True, scalar = True)
        F_stat = wald.statistic
        p_value = wald.pvalue
        df_num = wald.df_num
        df_denom = wald.df_denom

        tests = [F_stat, df_num, df_denom, p_value]
        final_tests.append(tests)

    # return a df instead of a 2d list
    results_table = pd.DataFrame(
    final_results,
    columns=[
        "Intercept",
        "Intercept p-value",
        "Slope",
        "Slope p-value",
        "Number of markets"
    ],
    index=time_labels[:len(final_results)]
    )

    tests_table = pd.DataFrame(
    final_tests,
    columns=[
        "F statistic",
        "df num",
        "df denom",
        "p-value",
    ],
    index=time_labels[:len(final_tests)]
    )   

    return results_table, tests_table

In [ ]:
final_results, final_tests = mincer_zarnowitz(panel)

In [ ]:
final_results

In [ ]:
final_tests

In [ ]:
# creates a subset of the data that can be put into compute_decay_curve 
def create_cohort(data, target_bin, time_edges = [0, 24, 48, 96, 168, 336, 720, 1440, 2880, np.inf], 
                time_labels = [
                "0-1 day",
                "1-2 days",
                "2-4 days",
                "4-7 days",
                "1-2 weeks",
                "2-4 weeks",
                "1-2 months",
                "2-4 months",
                ">4 months"]):
    # for each market, find the furthest point it was ever tracked
    max_ttr_per_market = data.groupby("market_id")["time_to_resolution"].max()

    # same time-binning as before
    market_lifespan_bin = pd.cut(
    max_ttr_per_market,
    bins=time_edges,
    labels=time_labels,
    include_lowest=True,
    ordered=True
    )

    # filter to find the bin that matches the target and select those from the input data
    matching_ids = market_lifespan_bin[market_lifespan_bin == target_bin].index
    cohort = data[data.index.get_level_values("market_id").isin(matching_ids)].copy()

    print(f"Cohort size: {cohort.index.get_level_values('market_id').nunique()} markets")
    print(f"Original sample: {data.index.get_level_values('market_id').nunique()} markets")

    return cohort

In [ ]:
cohort_panel = create_cohort(panel, ">4 months") # to control for market size
cohort_results, cohort_tests = mincer_zarnowitz(cohort_panel)

In [ ]:
cohort_results

In [ ]:
cohort_tests

In [ ]:
# same aggregation that we've been doing, but yes_rate, pred and n are now column names
df = panel.groupby("price_bins", observed=True).agg(
    yes_rate=("yes_outcomes", "mean"),
    pred=("p", "mean"),
    n=("p", "size"))

df.reset_index()

plt.plot([0, 1], [0, 1], "--", color="gray")
plt.scatter(df["pred"], df["yes_rate"])
plt.xlabel("Predicted probability")
plt.ylabel("Actual probability")
plt.show()

## Martingales

In [4]:
# a more memory efficient implementation
# doesnt store unnecessary copies of panel like the old one did
# and uses gc to clear memory as well

panel["log_hours_to_end"] = np.log1p(panel["hours_to_end"].clip(lower=0))
# dummy to take care of the impact of hours that are after the scheduled end date
panel["past_end"] = (panel["hours_to_end"] < 0).astype("int8")
panel.drop(columns=["hours_to_end"], inplace=True)

g = panel.groupby("market_id", observed=True) # groupby object
# multiplying by 100 makes 0.01 dollars into 1 cent, more interpretable
panel["delta_p"] = 100 * g["p"].diff()
panel["p_lag"]   = 100 * g["p"].shift(1)

# these are already in cent terms since its built from delta_p which is in cents now
panel["delta_p_lagged"]  = g["delta_p"].shift(1)
panel["delta_p_lagged2"] = g["delta_p"].shift(2)
panel["delta_p_lagged3"] = g["delta_p"].shift(3)
# 24 hour rolling std dev
panel["volatility"] = (g["delta_p_lagged"].rolling(24, min_periods=5).std()
                       .reset_index(level=0, drop=True))

# turns panel itself into the regression frame, without keeping a second copy to save mem
panel.drop(columns=["p"], inplace=True)
panel.dropna(inplace=True)
panel.reset_index(inplace=True)          # market_id becomes a column for clustering
del g                                    # the groupby object keeps a reference to the data
gc.collect();

In [7]:
# martingale
model = feols("delta_p ~ p_lag + np.log1p(time_to_resolution) + delta_p_lagged + volatility",
                 data = panel, 
                 vcov={"CRV1": "market_id"}, # cluster se by market id
                 copy_data=False, store_data=False, lean=True) # more memory optimizations

model.summary(digits = 6)

# # method = qr alleviates memory troubles
# results = model.fit(method="qr", cov_type="cluster", cov_kwds={"groups": market_ids})
# print(results.summary())

###

Estimation:  OLS
Dep. var.: delta_p
sample: None = all
Inference:  CRV1
Observations:  43001560

| Coefficient                  |   Estimate |   Std. Error |    t value |   Pr(>|t|) |      2.5% |     97.5% |
|:-----------------------------|-----------:|-------------:|-----------:|-----------:|----------:|----------:|
| Intercept                    |   0.087684 |     0.002672 |  32.810216 |   0.000000 |  0.082446 |  0.092922 |
| p_lag                        |  -0.000687 |     0.000016 | -41.670997 |   0.000000 | -0.000720 | -0.000655 |
| np.log1p(time_to_resolution) |  -0.010038 |     0.000385 | -26.081032 |   0.000000 | -0.010792 | -0.009283 |
| delta_p_lagged               |  -0.135257 |     0.001966 | -68.781406 |   0.000000 | -0.139111 | -0.131403 |
| volatility                   |  -0.016123 |     0.000519 | -31.090081 |   0.000000 | -0.017140 | -0.015107 |
---
RMSE: 1.811519 R2: 0.018677 


In [8]:
# annoying typechecking error
wald = model.wald_test()          # H0: Intercept = p_lag = ... = scale(volatility) = 0

print(f"""F_stat: {wald['statistic']}
p_value: {wald['pvalue']}
df_num: {model._dfn}
df_denom: {model._dfd}""")

F_stat: 1677.7931274198404
p_value: 0.0
df_num: 5
df_denom: 49559


In [5]:
# martingale w many lags
model = feols("delta_p ~ p_lag + log_hours_to_end + past_end"
            " + delta_p_lagged + delta_p_lagged2 + delta_p_lagged3 + volatility",
              data = panel,
              vcov={"CRV1": "market_id"},
              copy_data=False, store_data=False, lean=True)

model.summary(digits = 6)

# # method = qr alleviates memory troubles
# results = model.fit(method="qr", cov_type="cluster", cov_kwds={"groups": market_ids})
# print(results.summary())

###

Estimation:  OLS
Dep. var.: delta_p
sample: None = all
Inference:  CRV1
Observations:  42353513

| Coefficient      |   Estimate |   Std. Error |    t value |   Pr(>|t|) |      2.5% |     97.5% |
|:-----------------|-----------:|-------------:|-----------:|-----------:|----------:|----------:|
| Intercept        |   0.037389 |     0.001810 |  20.661885 |   0.000000 |  0.033842 |  0.040936 |
| p_lag            |  -0.000464 |     0.000013 | -34.548279 |   0.000000 | -0.000491 | -0.000438 |
| log_hours_to_end |  -0.003308 |     0.000239 | -13.838971 |   0.000000 | -0.003777 | -0.002840 |
| past_end         |   0.017722 |     0.004843 |   3.659576 |   0.000253 |  0.008230 |  0.027214 |
| delta_p_lagged   |  -0.142909 |     0.002129 | -67.133729 |   0.000000 | -0.147081 | -0.138737 |
| delta_p_lagged2  |  -0.050506 |     0.001253 | -40.292344 |   0.000000 | -0.052963 | -0.048049 |
| delta_p_lagged3  |  -0.014617 |     0.001252 | -11.676668 |   0.000000 | -0.017070 | -0.012163 |
| volat

In [6]:
# annoying typechecking error
wald = model.wald_test()          # H0: Intercept = p_lag = ... = scale(volatility) = 0

print(f"""F_stat: {wald['statistic']}
p_value: {wald['pvalue']}
df_num: {model._dfn}
df_denom: {model._dfd}""")

F_stat: 1291.4257905518552
p_value: 0.0
df_num: 7
df_denom: 49559


In [ ]:
# autocorrelation
cols_needed = ["delta_p", "delta_p_lagged", "delta_p_lagged2", "delta_p_lagged3"]
panel_reg = panel[cols_needed].dropna().copy()

model = smf.ols("delta_p ~ delta_p_lagged + delta_p_lagged2 + delta_p_lagged3", data = panel_reg)
market_ids = panel_reg.index.get_level_values("market_id")

# method = qr alleviates memory troubles
results = model.fit(method="qr", cov_type="cluster", cov_kwds={"groups": market_ids})
print(results.summary())

In [ ]:
wald = results.wald_test("Intercept = 0, delta_p_lagged = 0, delta_p_lagged2 = 0, delta_p_lagged3=0", use_f=True, scalar = True)
F_stat = wald.statistic
p_value = wald.pvalue
df_num = wald.df_num
df_denom = wald.df_denom

print(f"""F_stat: {F_stat}
p_value: {p_value}
df_num: {df_num}
df_denom: {df_denom}""")

In [ ]:
cols_needed = ["delta_p", "p", "time_to_resolution", "delta_p_lagged", "volatility"]
panel_reg = panel[cols_needed].dropna().copy()

model = smf.ols("delta_p ~ scale(p) + scale(time_to_resolution) + scale(delta_p_lagged) + scale(volatility)", data = panel_reg)
market_ids = panel_reg.index.get_level_values("market_id")

# method = qr alleviates memory troubles
results = model.fit(method="qr", cov_type="cluster", cov_kwds={"groups": market_ids})
print(results.summary())

In [ ]:
wald = results.wald_test("Intercept = 0, scale(p) = 0, scale(time_to_resolution) = 0, scale(delta_p_lagged) = 0, scale(volatility) = 0", use_f=True, scalar = True)
F_stat = wald.statistic
p_value = wald.pvalue
df_num = wald.df_num
df_denom = wald.df_denom

print([F_stat, p_value, df_num, df_denom])

In [ ]:
max_lag = 24

# plot the autocorrelation function
acf = []

for lag in range(1, max_lag + 1):
    # corr between delta p and delta_p_lagged for all lag values from 1 to 24
    corr = panel["delta_p"].corr(panel.groupby("market_id", observed=True)["delta_p"].shift(lag))
    acf.append(corr)

plt.figure(figsize=(8,4))
plt.plot(range(1, max_lag + 1), acf, marker="o")
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Lag (hours)")
plt.ylabel("Correlation")
plt.title("Panel autocorrelation of hourly price changes")
plt.show()

In [ ]:
print(panel.columns.tolist())

In [ ]:
print(panel.index.names)
print(panel.index.get_level_values("market_id").nunique())

In [ ]:
import statsmodels.api as sm
results = []

count = 0
for market_id, df in panel.groupby(level = "market_id"):
    df = df.dropna(subset=[
        "delta_p",
        "delta_p_lagged",
        "delta_p_lagged2",
        "delta_p_lagged3"
    ])

    # Require a reasonable sample size
    if len(df) < 30:
        continue

    X = sm.add_constant(
        df[[
            "delta_p_lagged",
            "delta_p_lagged2",
            "delta_p_lagged3"
        ]]
    )

    y = df["delta_p"]

    try:
        model = sm.OLS(y, X).fit()

        results.append({
            "market_id": market_id,
            "beta1": model.params["delta_p_lagged"],
            "beta2": model.params["delta_p_lagged2"],
            "beta3": model.params["delta_p_lagged3"],
            "r2": model.rsquared,
            "n": len(df)
        })

    except Exception as e:
        print(f"{market_id}: {e}")
        break

betas = pd.DataFrame(results)

print(betas.describe())

betas.to_csv("market_specific_ar3_coefficients.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))

for ax, col in zip(axes, ["beta1","beta2","beta3"]):

    x = betas[col]

    lo = x.quantile(0.01)
    hi = x.quantile(0.99)

    x = x[(x >= lo) & (x <= hi)]

    ax.hist(x, bins=75)
    ax.axvline(0, color="red", linestyle="--")
    ax.set_title(col)

plt.tight_layout()